# Introduction

In this notebook, I will work on a final modelisation to infer result for my thesis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from functionsFolder.config import text_data, biometric_data, adv_to_del, team_to_del, id, useless, index, to_keep
from functionsFolder.preProcessingAuto import preprocess_working_df
from functionsFolder.modelAutomation import get_features_and_target, clean_features, evaluate_models, prepare_features
from functionsFolder.optimisationModel import select_features, optimized_training
from sklearn.feature_selection import SelectPercentile
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from time import time
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
import joblib
import shap
from matplotlib.colors import LinearSegmentedColormap

##Different Paths
HOME = r"C:\Users\Utilisateur\Desktop\Master ULB\Mémoire"
W_DB = r"\Database\Working db"
STACK = r"\Thesis - Code\database\Database Updates\Database stack"

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Utilisateur\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
test_idf = pd.read_excel(r"C:\Users\Utilisateur\Desktop\Master ULB\Mémoire\Thesis - Code\tf_idf_df.xlsx")

In [3]:
team_to_avg = [
    col for col in test_idf.columns
    if ('team' in col or 'opp' in col) and 'text_' not in col and '%' not in col
]  #Creating a list with the columns to be divided

team_to_avg.remove('G_team_features_context') #Removing the number of games

test_idf[team_to_avg] = test_idf[team_to_avg].div(test_idf['G_team_features_context'], axis=0) #Dividing all the columns by the number of games

## To delete

In [6]:
dico_features_tfidf = {
    'Without' : ['Weight_features',
 'per_40_FG%_features',
 'per_40_3P%_features',
 'per_40_2P%_features',
 'per_40_FT%_features',
 'per_40_ORB_features',
 'per_40_TRB_features',
 'per_40_STL_features',
 'per_40_TOV_features',
 'advanced_PER_features',
 'advanced_3PAr_features',
 'advanced_FTr_features',
 'advanced_ORB%_features',
 'advanced_AST%_features',
 'advanced_STL%_features',
 'advanced_BLK%_features',
 'advanced_WS/40_features',
 'advanced_OBPM_features',
 'advanced_BPM_features'],
'With' : 
['Height_features',
 'Weight_features',
 'text_game_features_context',
 'per_40_FG_features',
 'per_40_FG%_features',
 'per_40_3P_features',
 'per_40_3PA_features',
 'per_40_3P%_features',
 'per_40_2PA_features',
 'per_40_2P%_features',
 'per_40_FT_features',
 'per_40_FTA_features',
 'per_40_FT%_features',
 'per_40_ORB_features',
 'per_40_DRB_features',
 'per_40_TRB_features',
 'per_40_AST_features',
 'per_40_STL_features',
 'per_40_BLK_features',
 'per_40_TOV_features',
 'per_40_PF_features',
 'G_team_features_context',
 'W-L%_features_context',
 'W.2_features_context',
 'Tm._features_context',
 'Opp._features_context',
 'FGA_team_features_context',
 '3PA_team_features_context',
 'FT_team_features_context',
 'FT%_team_features_context',
 'TRB_team_features_context',
 'PF_team_features_context',
 'FG_opp_features_context',
 'FGA_opp_features_context',
 '3P%_opp_features_context',
 'STL_opp_features_context',
 'PF_opp_features_context',
 'advanced_PER_features',
 'advanced_TS%_features',
 'advanced_3PAr_features',
 'advanced_FTr_features',
 'advanced_ORB%_features',
 'advanced_DRB%_features',
 'advanced_AST%_features',
 'advanced_OWS_features',
 'advanced_WS_features',
 'advanced_WS/40_features',
 'advanced_OBPM_features',
 'advanced_DBPM_features',
 'advanced_BPM_features']
}

## Add index

We need index for latter analysis

In [8]:
base_df = pd.read_excel(HOME + STACK + r"\looping_df_1746884524.8275206.xlsx")
RECOVERY_OPP = HOME + W_DB + r"\Features\Team data\27-4-2025_team.xlsx"

In [9]:
#add the mean of every position in the advance part and scouting evaluation part
base_df[to_keep['adv']] = base_df[to_keep['adv']].fillna(base_df.groupby('Pos_x_features')[to_keep['adv']].transform('median'))
base_df[to_keep['scouting_reports']] = base_df[to_keep['scouting_reports']].fillna(base_df.groupby('Pos_x_features')[to_keep['scouting_reports']].transform('median'))
processed_df = preprocess_working_df(base_df, recovery_file=RECOVERY_OPP)
#Get a copy of the dataset
working_df = processed_df.copy()
# Get initial features and targets
raw_features, target = get_features_and_target(working_df) #Get the features and target columns' name in separate lists


# Step 2: Prepare feature list (optionally re-adding 'adv' and 'scouting_reports')
include_subsets = ['per_40', 'team', 'opp', 'adv', 'scouting_reports']
features, included_types = prepare_features( #prepare the features' space
    base_features=raw_features,
    df=working_df,
    useless_list=useless, #list of features that we do not use 
    feature_dict=to_keep, #dictionnary of the features that we keep 
    include_keys=include_subsets #the keys to the dictionnary above
)
#Change the height from feet to inches 
working_df['Height_features'] = working_df['Height_features'].apply(
    lambda x: int(x.split('-')[0]) * 12 + int(x.split('-')[1])
)

#We keep only the players that where drafted after 2009 (60 players for 15 years = 900 max)
df = working_df[working_df['draft_season_features'] > 8].copy()
i = 1
df['med_tresh'] = (df[f'WS/48-{i}_target'] > 0).astype(int)

#We keep context variables in an list
keywords = ['team', 'opp', 'text']
filtered_list = [element for element in features if not any(keyword in element for keyword in keywords)]

2025-05-15 18:17:23,095 - INFO - Starting full preprocessing pipeline...
2025-05-15 18:17:23,097 - INFO - Creating dummy variables...
2025-05-15 18:17:23,097 - INFO - After dummy variable creation: shape = (1253, 389)
2025-05-15 18:17:23,097 - INFO - Categorizing and one-hot encoding categorical features...
2025-05-15 18:17:23,121 - INFO - After categorization and one-hot encoding: shape = (1253, 405)
2025-05-15 18:17:23,122 - INFO - Processing unstructured text data...
2025-05-15 18:17:26,786 - INFO - After TF-IDF vectorization: shape = (1253, 8291)
2025-05-15 18:17:26,836 - INFO - After after TF-IDF merge: shape = (1253, 8696)
2025-05-15 18:17:26,841 - INFO - Imputing opponent columns based on team data...
2025-05-15 18:17:29,012 - INFO - After imputation using team data: shape = (1253, 8696)
2025-05-15 18:17:29,014 - INFO - Imputing opponent columns using recovery data...
2025-05-15 18:17:36,095 - INFO - After imputation using recovery data: shape = (1253, 8696)
2025-05-15 18:17:36,

In [13]:
test_idf.set_index(df['player_id'])

,Height_features,Weight_features,dummy_hs_ranking_features_context,dummy_coll_awards_features_context,dummy_mock_draft_features_context,dummy_scouting_reports_features_context,mock_draft_1-25_features_context,mock_draft_26-50_features_context,mock_draft_51-75_features_context,mock_draft_76-100_features_context,...,Strength2_features_context,Quickness_features_context,Leadership_features_context,Jump Shot_features_context,NBA Ready_features_context,Rebounding_features_context,Potential_features_context,Post Skills_features_context,Intangibles_features_context,med_tresh
player_id,,,,,,,,,,,,,,,,,,,,,
aarongordon_15,81,225,1,1,1,1,1,0,0,0,...,7,8,8,6,8.0,7,8.0,6,9,1
aaronharrison_16,78,212,1,0,1,1,0,0,1,0,...,8,7,6,8,7.0,7,7.0,7,7,0
aaronhenry_22,78,210,0,0,1,1,0,0,1,0,...,8,8,7,7,8.0,6,7.0,7,7,0
aaronholiday_19,73,185,1,1,1,1,0,1,0,0,...,8,8,8,8,8.0,8,7.0,7,8,1
aaronjackson_18,80,215,0,0,0,0,0,0,0,0,...,7,8,8,8,8.0,8,7.0,7,8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zhairesmith_19,77,195,0,0,1,1,1,0,0,0,...,7,8,7,7,7.0,9,7.0,7,7,1
ziairewilliams_22,80,185,1,0,1,1,1,0,0,0,...,7,8,7,8,7.0,8,9.0,8,8,1
zionwilliamson_20,79,285,1,1,1,1,1,0,0,0,...,9,9,9,7,8.0,8,9.0,7,9,1


# TF-IDF Pipeline

In [4]:
features = [key for key in test_idf.columns if 'feature' in key] #All the features in a list 
filtered_list = [key for key in features if 'context' not in key] #The features without the contextual one in another

In [8]:
#Configuration
config = {
    'test_size': 0.2,
    'n_folds': 5, #CV=5
    'n_iter' : 200, 
    'scoring': 'roc_auc', #Optimize the AUC
    'features_max': 100,
    'features_sets' : { #To automatically change the dataset from contextual to not
        'With': features,
        #'Without' : filtered_list
    }
}

#dico_features_tfidf = {} #dictionnary for to features retain in select_features function

#X & y
y = test_idf['med_tresh']
X_raw = test_idf[features] #Full dataset with every features


#Base pipeline
base_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('resampler', SMOTETomek(random_state=143)),
    ('classifier', None)
])

#Classification model definition
classifiers = {
    'xgboost': XGBClassifier(random_state=48, 
                             eval_metric='auc'),
    'logistic_regression': LogisticRegression(random_state=42, 
                                              penalty='elasticnet',
                                              solver='saga'),
    'random_forest': RandomForestClassifier(random_state=42, 
                                            max_samples=0.7),
    'svm': SVC(random_state=42, 
               probability=True)
}

#Parameters search space
search_spaces = {
    'xgboost': {
        'classifier__learning_rate': Real(0.01, 0.2, prior='log-uniform'),
        'classifier__max_depth': Integer(2, 4),
        'classifier__n_estimators': Integer(100, 500),
        'classifier__subsample': Real(0.6, 1.0, 'uniform'),
        'classifier__colsample_bytree': Real(0.6, 1.0, 'uniform'),
        'classifier__gamma': Real(0, 1),
        'resampler__sampling_strategy': Real(0.5, 0.7)
    },
    'logistic_regression': {
        'classifier__C': Real(0.01, 10.0, prior='log-uniform'),
        'classifier__l1_ratio': Real(0.3, 0.7, 'uniform'),
        'resampler__sampling_strategy': Real(0.5, 0.7)
    },
    'random_forest': {
        'classifier__n_estimators': Integer(100, 500),
        'classifier__max_depth': Integer(2, 4),
        'classifier__min_samples_split': Integer(5, 20),
        'classifier__max_features': Categorical(['sqrt', 0.5]),
        'resampler__sampling_strategy': Real(0.5, 0.7)
    },
    'svm': {
        'classifier__C': Real(0.5, 50.0, prior='log-uniform'),
        'classifier__gamma': Categorical(['scale', 'auto'] + list(np.logspace(-3, -1, 5))),
        'classifier__kernel': Categorical(['rbf', 'linear']),
        'resampler__sampling_strategy': Real(0.5, 0.7)
    }
}

final_results = []
for features_set_name, context in config['features_sets'].items():
    X = X_raw[context]
    X = X_raw[dico_features_tfidf['With']]
    X_train_selected, X_test_selected, y_train, y_test = train_test_split(
        X, y, 
        test_size=config['test_size'],
        stratify=y,
        random_state=42
    )
    #X_train_selected, X_test_selected, features_selected, best_max_features = select_features(
     #   X_train, X_test, y_train,
      #  max_features_range=(25, 50),
       # n_iter=config['n_iter'])

    #dico_features_tfidf[features_set_name] = features_selected

    df_results = optimized_training(
        X_train_selected, X_test_selected, 
        y_train, y_test,
        classifiers, config, 
        'randomforest', search_spaces,
        'With'
    )

    final_results.append(df_results)

final_df = pd.concat(final_results)
final_df

c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\ma\core.py:2892: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\ma\core.py:2892: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\ma\core.py:2892: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\ma\core.py:2892: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge

,Features Set,Model,Train Accuracy,Test Accuracy,F1,Best AUC,Training Time (min),Nb features,Best Params
0,With,xgboost,0.780521,0.754098,0.839858,0.812654,15.9,50,"{'classifier__colsample_bytree': 0.6, 'classif..."
1,With,logistic_regression,0.750343,0.732240,0.835017,0.751698,11.7,50,"{'classifier__C': 0.12412608591232702, 'classi..."
2,With,random_forest,0.803841,0.781421,0.863014,0.812963,13.6,50,"{'classifier__max_depth': 4, 'classifier__max_..."
3,With,svm,0.711934,0.732240,0.810811,0.750154,32.8,50,"{'classifier__C': 3.630462491987826, 'classifi..."


In [ ]:
for i in final_df['Best Params']:
    print(i)

OrderedDict({'classifier__colsample_bytree': 0.6, 'classifier__gamma': 0.5520431082848759, 'classifier__learning_rate': 0.022573473201075948, 'classifier__max_depth': 2, 'classifier__n_estimators': 100, 'classifier__subsample': 0.6, 'resampler__sampling_strategy': 0.7})
OrderedDict({'classifier__C': 0.12412608591232702, 'classifier__l1_ratio': 0.5279519866673492, 'resampler__sampling_strategy': 0.5035871078493812})
OrderedDict({'classifier__max_depth': 4, 'classifier__max_features': 0.5, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 100, 'resampler__sampling_strategy': 0.6402097160127685})
OrderedDict({'classifier__C': 3.630462491987826, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear', 'resampler__sampling_strategy': 0.7})


## Cross Validation (CV=5)

In [45]:
##Contingency Table of the best model 
#1 y creation 
y = test_idf[test_idf['flag_reports_features_context']==1]['med_tresh']

#2 X Creation
X = test_idf[test_idf['flag_reports_features_context']==1][dico_features_tfidf['With']] #Full dataset with every features

X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=config['test_size'],
        stratify=y,
        random_state=42
    )

## Base pipeline
base_pipeline = ImbPipeline([
            ('scaler', StandardScaler()),
            ('resampler', SMOTETomek(sampling_strategy=0.7, random_state=143)),
            ('classifier', XGBClassifier(random_state=48,
                             colsample_bytree=0.6,
                             gamma= 0.5520431082848759,
                             learning_rate=0.022573473201075948,
                             max_depth=2,
                             n_estimator=100,
                             subsample=0.6,
                             eval_metric='auc')
            )
        ])

base_pipeline.fit(X_train, y_train)
train_pred = base_pipeline.predict(X_train)
test_pred = base_pipeline.predict(X_test)
train_proba = base_pipeline.predict_proba(X_train)[:, 1]
test_proba = base_pipeline.predict_proba(X_test)[:, 1]
y_pred = base_pipeline.predict(X_test)
train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)
f1 = f1_score(y_test, test_pred)
auc = roc_auc_score(y_test, test_proba)
print(f'test acc: {test_acc}, train acc: {train_acc}, f1: {f1}, auc: {auc}')


test acc: 0.7661290322580645, train acc: 0.8245967741935484, f1: 0.8527918781725888, auc: 0.6631578947368422


c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:43:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


### With the contextual data

In [50]:
best_classifiers = {
    'xgboost': {'model' : XGBClassifier(random_state=48,
                             colsample_bytree=0.6,
                             gamma= 0.5520431082848759,
                             learning_rate=0.022573473201075948,
                             max_depth=2,
                             n_estimator=100,
                             subsample=0.6,
                             eval_metric='auc'), 
                'resampler' : 0.7
            },
    'logistic_regression': {'model' : LogisticRegression(random_state=42, 
                                              penalty='elasticnet',
                                              solver='saga',
                                              C=0.12412608591232702,
                                              l1_ratio=0.5279519866673492
                                              ),
                            'resampler' : 0.5035871078493812,
            },
    'random_forest': {'model' : RandomForestClassifier(random_state=42, 
                                            max_samples=0.7,
                                            max_depth=4,
                                            max_features=0.5,
                                            min_samples_split=10,
                                            n_estimators=100
                                            ),
                    'resampler' : 0.6402097160127685
            },
    'svm': {'model' : SVC(random_state=42, 
               probability=True,
               C=3.630462491987826,
               gamma='scale',
               kernel='linear'
               ),
            'resampler' : 0.7
        }
}

In [ ]:
#5 Folds because 20% of the data
n_folds = 5
stratified_kfold = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
results = {}
y = test_idf['med_tresh']

X = test_idf[dico_features_tfidf['With']]
for name, model in best_classifiers.items():
    train_accuracy = []
    test_accuracy = []
    f1 = []
    auc = []
    
    best_pipeline = ImbPipeline([
            ('scaler', StandardScaler()),
            ('resampler', SMOTETomek(sampling_strategy=model['resampler'], random_state=143)),
            ('classifier', model['model'])
        ])


    for fold, (train_index, test_index) in enumerate(stratified_kfold.split(X, y)):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]

        #train the model
        best_pipeline.fit(X_train, y_train) 

        #Get the preidction 
        train_pred = best_pipeline.predict(X_train)
        test_pred = best_pipeline.predict(X_test)
        train_proba = best_pipeline.predict_proba(X_train)[:, 1]
        test_proba = best_pipeline.predict_proba(X_test)[:, 1]
              

        #Performances evaluation for this split
        accuracy = accuracy_score(y_test, test_pred)
        train_acc = accuracy_score(y_train, train_pred)
        test_acc = accuracy_score(y_test, test_pred)
        f1_s = f1_score(y_test, test_pred)
        auc_s = roc_auc_score(y_test, test_proba)   

        test_accuracy.append(test_acc)
        train_accuracy.append(train_acc)
        f1.append(f1_s)
        auc.append(auc_s)

    #Put the results in a dictionnary
    results[name] = {
        'mean_train_accuracy': np.median(train_accuracy),
        'mean_test_accuracy': np.median(test_accuracy),
        'mean_f1': np.median(f1),
        'mean_auc': np.median(auc),
        'all_auc_scores': auc,
        }

#Display the results
print("\nRésultats Agrégés:")
for name, result in results.items():
    print(f"Modèle: {name}")
    print(f"  Mean Train Accuracy: {result['mean_train_accuracy']:.4f}")
    print(f"  Mean Test Accuracy: {result['mean_test_accuracy']:.4f}")
    print(f"  Mean F1: {result['mean_f1']:.4f}")
    print(f"  Mean AUC: {result['mean_auc']:.4f}")
    print("-" * 40)

c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:50:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:50:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:50:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:50:52] WARNING: C:


Résultats Agrégés:
Modèle: xgboost
  Mean Train Accuracy: 0.7750
  Mean Test Accuracy: 0.6995
  Mean F1: 0.8000
  Mean AUC: 0.6722
----------------------------------------
Modèle: logistic_regression
  Mean Train Accuracy: 0.7613
  Mean Test Accuracy: 0.7432
  Mean F1: 0.8351
  Mean AUC: 0.6944
----------------------------------------
Modèle: random_forest
  Mean Train Accuracy: 0.8107
  Mean Test Accuracy: 0.7253
  Mean F1: 0.8214
  Mean AUC: 0.6753
----------------------------------------
Modèle: svm
  Mean Train Accuracy: 0.7315
  Mean Test Accuracy: 0.6831
  Mean F1: 0.7661
  Mean AUC: 0.6831
----------------------------------------


### Without the contextual data

In [62]:
best_classifiers_wo = {
    'xgboost': {'model' : XGBClassifier(random_state=48,
                             colsample_bytree=0.7092597368198831,
                             gamma= 1.0,
                             learning_rate=0.01,
                             max_depth=5,
                             n_estimator=100,
                             subsample=0.6,
                             eval_metric='auc'), 
                'resampler' : 0.7
            },
    'logistic_regression': {'model' : LogisticRegression(random_state=42, 
                                              penalty='elasticnet',
                                              solver='saga',
                                              C=1.1353428950266315,
                                              l1_ratio=0.3003997149844717
                                              ),
                            'resampler' : 0.6279667955954684,
            },
    'random_forest': {'model' : RandomForestClassifier(random_state=42, 
                                            max_samples=0.7,
                                            max_depth=6,
                                            max_features=0.5,
                                            min_samples_split=19,
                                            n_estimators=500
                                            ),
                    'resampler' : 0.6167482675797911
            },
    'svm': {'model' : SVC(random_state=42, 
               probability=True,
               C=6.340874614693904,
               gamma=0.001,
               kernel='rbf'
               ),
            'resampler' : 0.7
        }
}

In [63]:
#5 Folds because 20% of the data
n_folds = 5
stratified_kfold = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
results = {}
y = test_idf['med_tresh']

X = test_idf[dico_features_tfidf['Without']]
for name, model in best_classifiers_wo.items():
    train_accuracy = []
    test_accuracy = []
    f1 = []
    auc = []
    
    best_pipeline = ImbPipeline([
            ('scaler', StandardScaler()),
            ('resampler', SMOTETomek(sampling_strategy=model['resampler'], random_state=143)),
            ('classifier', model['model'])
        ])


    for fold, (train_index, test_index) in enumerate(stratified_kfold.split(X, y)):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]

        #train the model
        best_pipeline.fit(X_train, y_train) 

        #Get the preidction 
        train_pred = best_pipeline.predict(X_train)
        test_pred = best_pipeline.predict(X_test)
        train_proba = best_pipeline.predict_proba(X_train)[:, 1]
        test_proba = best_pipeline.predict_proba(X_test)[:, 1]
              

        #Performances evaluation for this split
        accuracy = accuracy_score(y_test, test_pred)
        train_acc = accuracy_score(y_train, train_pred)
        test_acc = accuracy_score(y_test, test_pred)
        f1_s = f1_score(y_test, test_pred)
        auc_s = roc_auc_score(y_test, test_proba)   

        test_accuracy.append(test_acc)
        train_accuracy.append(train_acc)
        f1.append(f1_s)
        auc.append(auc_s)

    #Put the results in a dictionnary
    results[name] = {
        'mean_train_accuracy': np.median(train_accuracy),
        'mean_test_accuracy': np.median(test_accuracy),
        'mean_f1': np.median(f1),
        'mean_auc': np.median(auc),
        'all_auc_scores': auc,
        }

#Display the results
print("\nRésultats Agrégés:")
for name, result in results.items():
    print(f"Modèle: {name}")
    print(f"  Mean Train Accuracy: {result['mean_train_accuracy']:.4f}")
    print(f"  Mean Test Accuracy: {result['mean_test_accuracy']:.4f}")
    print(f"  Mean F1: {result['mean_f1']:.4f}")
    print(f"  Mean AUC: {result['mean_auc']:.4f}")
    print("-" * 40)

c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:59:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:59:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:59:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "n_estimator" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Utilisateur\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [10:59:25] WARNING: C:


Résultats Agrégés:
Modèle: xgboost
  Mean Train Accuracy: 0.8466
  Mean Test Accuracy: 0.7268
  Mean F1: 0.8276
  Mean AUC: 0.6549
----------------------------------------
Modèle: logistic_regression
  Mean Train Accuracy: 0.7160
  Mean Test Accuracy: 0.7268
  Mean F1: 0.8188
  Mean AUC: 0.6961
----------------------------------------
Modèle: random_forest
  Mean Train Accuracy: 0.8493
  Mean Test Accuracy: 0.7213
  Mean F1: 0.8235
  Mean AUC: 0.6673
----------------------------------------
Modèle: svm
  Mean Train Accuracy: 0.7151
  Mean Test Accuracy: 0.7158
  Mean F1: 0.8030
  Mean AUC: 0.7057
----------------------------------------


## Results analysis

### Confusion matrix

I will look at the results from the confusion for the best model out of 8

In [ ]:
#Create the confusion matrix
cm = confusion_matrix(y_test, y_pred, normalize='true')

#Get the name of the classes rather than 0 and 1
labels = ['Negative', 'Positive']

#Convertion to a dataframe
cm_df = pd.DataFrame(cm, index=[f'Actual {label}' for label in labels],
                         columns=[f'Predicted {label}' for label in labels])

# Palette 1 : Philadelphia 76ers (bleu royal) - with ChatGPT
sixers_colors = ['#E6EFFF', '#A3BFFA', '#0047AB', '#003087']  # Très pâle, pâle, bleu moyen, bleu royal
sixers_cmap = LinearSegmentedColormap.from_list("Sixers", sixers_colors)

#Display the matrix
def plot_confusion_matrix(cm_df, cmap, title):
    plt.figure(figsize=(8, 6), dpi=360)
    sns.heatmap(cm_df, annot=True, fmt='.2f', cmap=cmap, cbar=True,
                annot_kws={'size': 12, 'weight': 'bold'}, linewidths=0.5, 
                linecolor='white')
    plt.title(title, fontsize=14, pad=15)
    plt.xlabel('Predictions', fontsize=12)
    plt.ylabel('Real Values', fontsize=12)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(cm_df, sixers_cmap, 'Confusion Matrix from XGBoost - Norm true')

### Shapley values

I will look at the shapley values for the best model 

In [ ]:
#Do not work with ImbPipeline, hence we redo the train manually
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)
sm = SMOTETomek(sampling_strategy = 0.7, random_state=143)
X_resampled, y_resampled = sm.fit_resample(X_train_scaled, y_train)

model = XGBClassifier(learning_rate=0.01,
                        max_depth=7,
                        n_estimators=298,
                        subsample=0.6804196802009144,
                        colsample_bytree=0.6,
                        gamma=0.0,
                        random_state=48)
model.fit(X_resampled, y_resampled)

#Shap values calculation
explainer = shap.Explainer(model, X_resampled)
shap_values = explainer(X_test_scaled_df)

#Beeswarm plot
shap.plots.beeswarm(shap_values, max_display=10)  #10 to keep lisibility

### Do we improve GMs decisions ? 

In [ ]:
player_names = X_test.index

#Get a df with the results from the prediction (of the test set)
results = pd.DataFrame({
    'Player': player_names,
    'Predicted': y_pred,
    'Actual': y_test
})

results['Results'] = results['Predicted'] == results['Actual'] #
results

In [ ]:
results[results['Results'] == False]